# Reviewer benchmark: read-only results analysis

This notebook consumes the machine-readable outputs of `src/reviewer_benchmark.py`. It never tokenizes sequences, constructs features, or fits a model. Set `REVIEWER_BENCHMARK_OUTPUT` to the output directory (default: `results/reviewer_benchmark`) before running. Empty or partial output directories are expected while experiments are pending; cells report what is missing and continue.

The primary unit is an out-of-fold condition. Prefix distributions therefore include every prespecified prefix, while full-versus-filtered comparisons use the same fold-level metrics and resource metadata.

In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    PLOTTING_AVAILABLE = True
except ImportError:
    PLOTTING_AVAILABLE = False

SRC = Path.cwd() / "src"
if SRC.is_dir() and str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
try:
    from utilities.benchmark_metrics import (
        chance_balanced_accuracy, fold_confidence_interval,
        paired_difference_confidence_interval, prediction_metrics,
    )
except ImportError:
    def chance_balanced_accuracy(n_classes: int) -> float:
        return 1.0 / int(n_classes)
    prediction_metrics = None
    fold_confidence_interval = None
    paired_difference_confidence_interval = None

OUTPUT_ROOT = Path(os.environ.get("REVIEWER_BENCHMARK_OUTPUT", "results/reviewer_benchmark"))
SPLIT_MANIFEST_PATH = os.environ.get("REVIEWER_BENCHMARK_SPLIT_MANIFEST")
EXPORT_OUTPUTS = os.environ.get("REVIEWER_BENCHMARK_EXPORT", "0") == "1"
EXPORT_DIR = Path(os.environ.get("REVIEWER_BENCHMARK_EXPORT_DIR", str(OUTPUT_ROOT / "publication_exports")))
CONFIDENCE = 0.95
BOOTSTRAP_SEED = 20260901
print(f"Reading saved benchmark outputs from: {OUTPUT_ROOT.resolve()}")
print("Plotting available:", PLOTTING_AVAILABLE, "| Export enabled:", EXPORT_OUTPUTS)

## 1. Load and validate manifests, metrics, predictions, and resource tables

All tables are discovered recursively so both a consolidated run and one-directory-per-condition output are supported. JSON manifests that contain configuration dictionaries are retained separately from tabular manifests.

In [ ]:
TABLE_EXTENSIONS = {".csv", ".parquet", ".pq", ".json"}

def _read_table(path: Path) -> pd.DataFrame | None:
    try:
        if path.suffix.lower() in {".parquet", ".pq"}:
            return pd.read_parquet(path)
        if path.suffix.lower() == ".csv":
            return pd.read_csv(path)
        if path.suffix.lower() == ".json":
            value = json.loads(path.read_text())
            if isinstance(value, list):
                return pd.DataFrame(value)
            if isinstance(value, dict):
                # Failure records are scalar-key JSON objects; loading
                # them as one row keeps failed conditions visible in QC.
                # Nested resolved manifests are excluded by _discover's
                # filename patterns and are loaded separately below.
                return pd.DataFrame(value) if all(isinstance(item, list) for item in value.values()) else pd.DataFrame([value])
    except Exception as error:
        print(f"Could not read {path}: {type(error).__name__}: {error}")
    return None

def _discover(patterns: tuple[str, ...]) -> list[Path]:
    if not OUTPUT_ROOT.exists():
        return []
    export_root = EXPORT_DIR.resolve()
    paths = [p for p in OUTPUT_ROOT.rglob("*") if p.is_file() and p.suffix.lower() in TABLE_EXTENSIONS and export_root not in p.resolve().parents]
    return sorted({p for p in paths if any(pattern in str(p).lower() for pattern in patterns)})

def _concat_named(patterns: tuple[str, ...], name: str) -> pd.DataFrame:
    frames = []
    for path in _discover(patterns):
        frame = _read_table(path)
        if frame is not None and not frame.empty:
            frame = frame.copy()
            frame["_source_file"] = str(path)
            frames.append(frame)
    result = pd.concat(frames, ignore_index=True, sort=False) if frames else pd.DataFrame()
    print(f"{name}: {len(result):,} rows from {len(frames)} file(s)")
    return result

metrics = _concat_named(("metrics",), "Metrics")
resources = _concat_named(("resources",), "Resources")
predictions = _concat_named(("predictions",), "Predictions")
conditions = _concat_named(("conditions",), "Condition manifests")
split_manifest = _concat_named(("split", "fold", "cluster"), "Split manifests")
if SPLIT_MANIFEST_PATH and Path(SPLIT_MANIFEST_PATH).exists():
    extra_split = _read_table(Path(SPLIT_MANIFEST_PATH))
    if extra_split is not None:
        extra_split = extra_split.copy()
        extra_split["_source_file"] = str(Path(SPLIT_MANIFEST_PATH))
        split_manifest = pd.concat([split_manifest, extra_split], ignore_index=True, sort=False)
        print(f"Loaded split manifest from REVIEWER_BENCHMARK_SPLIT_MANIFEST: {len(extra_split):,} rows")
prefix_manifest = _concat_named(("prefix",), "Prefix manifests")
failures = _concat_named(("failure",), "Failure records")
resfinder = _concat_named(("resfinder",), "Optional ResFinder tables")
# Keep resolved run manifests available for provenance without treating their
# nested configuration dictionaries as tidy result tables.
resolved_manifests = []
for manifest_path in sorted(OUTPUT_ROOT.rglob("*manifest*.json")) if OUTPUT_ROOT.exists() else []:
    try:
        value = json.loads(manifest_path.read_text())
        if isinstance(value, dict):
            resolved_manifests.append({"path": str(manifest_path), **value})
    except Exception as error:
        print(f"Could not read manifest {manifest_path}: {error}")
print(f"Resolved JSON manifests: {len(resolved_manifests)}")

def _validate(frame: pd.DataFrame, required: set[str], name: str) -> bool:
    missing = sorted(required.difference(frame.columns))
    if frame.empty:
        print(f"{name}: empty (this is expected before experiments finish)")
        return False
    if missing:
        print(f"{name}: missing required columns {missing}; downstream summaries may be unavailable")
        return False
    print(f"{name}: schema OK ({len(frame.columns)} columns)")
    return True

metrics_ok = _validate(metrics, {"condition_id", "fold_id", "balanced_accuracy"}, "Metrics")
resources_ok = _validate(resources, {"condition_id", "stage"}, "Resources")
predictions_ok = _validate(predictions, {"condition_id", "genome_id", "fold_id", "true_label", "predicted_label"}, "Predictions")
_validate(conditions, {"condition_id", "phenotype", "representation", "total_k"}, "Conditions")
_validate(split_manifest, {"genome_id", "fold_id"}, "Split manifest")


In [ ]:
# Normalize shared condition fields while preserving every source row.
CONDITION_COLUMNS = ["condition_id", "model_name", "dataset", "phenotype", "representation", "total_k", "prefix", "prefix_length", "suffix_length", "prefix_source", "positive_class"]
def _condition_fields(frame: pd.DataFrame) -> pd.DataFrame:
    frame = frame.copy()
    for column in CONDITION_COLUMNS:
        if column not in frame:
            frame[column] = pd.NA
    if "total_k" in frame:
        frame["total_k"] = pd.to_numeric(frame["total_k"], errors="coerce")
    if "prefix_length" in frame:
        frame["prefix_length"] = pd.to_numeric(frame["prefix_length"], errors="coerce")
    return frame

metrics = _condition_fields(metrics)
resources = _condition_fields(resources)
predictions = _condition_fields(predictions)
conditions = _condition_fields(conditions)
for frame in [metrics, resources, predictions]:
    frame["model_name"] = frame["model_name"].fillna("logistic_regression")
if not metrics_ok and metrics.empty:
    print("No metrics yet: later cells will show empty-data messages and create no figures.")

## 2. Dataset summary

This summary is derived from saved predictions and split metadata. It reports genomes, classes, class balance, folds, clusters, and high-level organism metadata when such columns were included in a manifest.

In [ ]:
def dataset_summary(prediction_table: pd.DataFrame, split_table: pd.DataFrame) -> pd.DataFrame:
    if prediction_table.empty:
        return pd.DataFrame(columns=["dataset", "phenotype", "genomes", "classes", "class_balance", "folds", "clusters"])
    rows = []
    for keys, group in prediction_table.groupby(["dataset", "phenotype"], dropna=False):
        dataset, phenotype = keys
        labels = group[["genome_id", "true_label"]].drop_duplicates("genome_id")
        counts = labels["true_label"].value_counts(dropna=False).to_dict()
        ids = set(labels["genome_id"].astype(str))
        split = split_table[split_table["genome_id"].astype(str).isin(ids)] if not split_table.empty and "genome_id" in split_table else pd.DataFrame()
        organism_columns = [c for c in ["species", "organism", "genus", "taxon", "organism_name"] if c in split]
        organism_composition = {c: {str(k): int(v) for k, v in split[c].dropna().value_counts().head(20).items()} for c in organism_columns}
        rows.append({
            "dataset": dataset, "phenotype": phenotype, "genomes": len(labels),
            "classes": len(counts), "class_balance": json.dumps({str(k): int(v) for k, v in counts.items()}),
            "folds": split["fold_id"].nunique() if "fold_id" in split else group["fold_id"].nunique(),
            "clusters": split["cluster_id"].nunique() if "cluster_id" in split else pd.NA,
            "organism_composition": json.dumps(organism_composition) if organism_composition else pd.NA,
        })
    return pd.DataFrame(rows)

dataset_table = dataset_summary(predictions, split_manifest)
display(dataset_table)
if dataset_table.empty:
    print("Dataset summary unavailable until prediction or split-manifest outputs exist.")

## 3. Full versus prefix-filtered performance

Fold-level scores remain the source of truth. The summary below computes condition-level means, medians, standard deviations, and percentile intervals from those tidy rows; no plotted value is entered as a constant. Chance balanced accuracy is `1 / n_classes`.

In [ ]:
def _percentile_interval(values: pd.Series, confidence: float = CONFIDENCE) -> tuple[float, float]:
    values = pd.to_numeric(values, errors="coerce").dropna().to_numpy(dtype=float)
    if not len(values):
        return (np.nan, np.nan)
    alpha = (1 - confidence) / 2
    return (float(np.quantile(values, alpha)), float(np.quantile(values, 1 - alpha)))

def summarize_performance(metric_table: pd.DataFrame) -> pd.DataFrame:
    required = {"condition_id", "fold_id", "balanced_accuracy"}
    if metric_table.empty or not required.issubset(metric_table.columns):
        return pd.DataFrame()
    table = _condition_fields(metric_table)
    rows = []
    grouping = ["condition_id", "model_name", "dataset", "phenotype", "representation", "total_k", "prefix", "prefix_length", "prefix_source"]
    for keys, group in table.groupby(grouping, dropna=False):
        if fold_confidence_interval is not None:
            seed = BOOTSTRAP_SEED + sum(ord(char) for char in str(keys[0]))
            interval = fold_confidence_interval(group, value_column="balanced_accuracy", seed=seed)
            lo, hi = interval["ci_lower"], interval["ci_upper"]
        else:
            lo, hi = _percentile_interval(group["balanced_accuracy"])
        classes = pd.to_numeric(group.get("n_classes", pd.Series(dtype=float)), errors="coerce").dropna()
        n_classes = int(classes.iloc[0]) if len(classes) else int(group.get("true_label", pd.Series(dtype=object)).nunique()) or 2
        row = dict(zip(grouping, keys))
        row.update({"n_folds": group["fold_id"].nunique(), "balanced_accuracy_mean": group["balanced_accuracy"].mean(), "balanced_accuracy_median": group["balanced_accuracy"].median(), "balanced_accuracy_std": group["balanced_accuracy"].std(ddof=1), "ci_lower": lo, "ci_upper": hi, "n_classes": n_classes, "chance_balanced_accuracy": chance_balanced_accuracy(n_classes)})
        for metric in ["sensitivity", "specificity", "macro_f1", "roc_auc"]:
            values = pd.to_numeric(group.get(metric, pd.Series(dtype=float)), errors="coerce")
            row[f"{metric}_mean"] = values.mean()
            if fold_confidence_interval is not None and values.notna().any():
                metric_interval = fold_confidence_interval(values.dropna().to_numpy(), seed=seed + sum(ord(char) for char in metric))
                row[f"{metric}_ci_lower"] = metric_interval["ci_lower"]
                row[f"{metric}_ci_upper"] = metric_interval["ci_upper"]
        rows.append(row)
    return pd.DataFrame(rows)

performance = summarize_performance(metrics)
display(performance.head(20))
if performance.empty:
    print("Performance summaries will appear after condition metrics are saved.")

def paired_difference_table(metric_table: pd.DataFrame, summary_table: pd.DataFrame) -> pd.DataFrame:
    """Align full and filtered fold scores by fold ID and compute full-minus-filtered intervals."""
    if metric_table.empty or summary_table.empty:
        return pd.DataFrame()
    rows = []
    filtered_conditions = summary_table[summary_table["representation"].eq("prefix_filtered")]
    for condition in filtered_conditions.itertuples(index=False):
        same = metric_table[(metric_table["model_name"] == condition.model_name) & (metric_table["dataset"] == condition.dataset) & (metric_table["phenotype"] == condition.phenotype) & (metric_table["total_k"] == condition.total_k)]
        full = same[same["representation"].eq("full")]
        current = same[same["condition_id"].eq(condition.condition_id)]
        if full.empty or current.empty: continue
        aligned = full[["fold_id", "balanced_accuracy"]].merge(current[["fold_id", "balanced_accuracy"]], on="fold_id", suffixes=("_full", "_filtered"), validate="one_to_one")
        if aligned.empty: continue
        differences = aligned["balanced_accuracy_full"] - aligned["balanced_accuracy_filtered"]
        if paired_difference_confidence_interval is not None:
            full_values = aligned[["fold_id", "balanced_accuracy_full"]].rename(columns={"balanced_accuracy_full": "score"})
            filtered_values = aligned[["fold_id", "balanced_accuracy_filtered"]].rename(columns={"balanced_accuracy_filtered": "score"})
            interval = paired_difference_confidence_interval(full_values, filtered_values, id_column="fold_id", value_column="score", seed=BOOTSTRAP_SEED)
            lo, hi = interval["ci_lower"], interval["ci_upper"]
        else:
            lo, hi = _percentile_interval(differences)
        rows.append({"condition_id": condition.condition_id, "model_name": condition.model_name, "dataset": condition.dataset, "phenotype": condition.phenotype, "total_k": condition.total_k, "prefix": condition.prefix, "prefix_length": condition.prefix_length, "paired_n_folds": len(aligned), "paired_difference_mean": differences.mean(), "paired_difference_median": differences.median(), "paired_ci_lower": lo, "paired_ci_upper": hi})
    return pd.DataFrame(rows)

paired_difference_summary = paired_difference_table(metrics, performance)
display(paired_difference_summary.head(20))
if paired_difference_summary.empty and not performance.empty:
    print("Paired intervals will appear when matching full and filtered fold rows are available.")

## 4. Prefix robustness

Each seeded prefix is an observation. Historical prefixes are overlaid only at their matching length and are assigned a descriptive percentile against the seeded distribution for the same phenotype and `total_k`; they are never used to select a test-set winner.

In [ ]:
def prefix_robustness_table(summary: pd.DataFrame) -> pd.DataFrame:
    if summary.empty or "representation" not in summary:
        return pd.DataFrame()
    filtered = summary[summary["representation"].eq("prefix_filtered")].copy()
    if filtered.empty:
        return filtered
    filtered["prefix_role"] = np.where(filtered["prefix_source"].astype(str).str.contains("historical", na=False), "historical", "seeded")
    random = filtered[filtered["prefix_role"].eq("seeded")]
    percentiles = []
    for _, row in filtered.iterrows():
        reference = random[(random["model_name"] == row["model_name"]) & (random["phenotype"] == row["phenotype"]) & (random["total_k"] == row["total_k"]) & (random["prefix_length"] == row["prefix_length"])]["balanced_accuracy_mean"]
        value = row["balanced_accuracy_mean"]
        percentiles.append(float((reference <= value).mean() * 100) if len(reference) else np.nan)
    filtered["historical_percentile"] = percentiles
    return filtered

prefix_summary = prefix_robustness_table(performance)
prefix_distribution = (prefix_summary[prefix_summary["prefix_role"].eq("seeded")] if not prefix_summary.empty else pd.DataFrame())
display(prefix_summary.head(30))
if prefix_summary.empty:
    print("Prefix robustness is unavailable until filtered-condition metrics exist.")

## 5. Accuracy versus compression, peak RAM, and runtime

Resource rows are stage measurements. Runtime is summed across stages and peak RSS is the maximum observed stage RSS. Repeated condition metadata are de-duplicated before joining to performance.

In [ ]:
def summarize_resources(resource_table: pd.DataFrame) -> pd.DataFrame:
    if resource_table.empty or "condition_id" not in resource_table:
        return pd.DataFrame()
    table = _condition_fields(resource_table)
    rows = []
    for keys, group in table.groupby(["condition_id", "model_name"], dropna=False):
        condition_id, model_name = keys
        row = {"condition_id": condition_id, "model_name": model_name}
        for column in ["dataset", "phenotype", "representation", "total_k", "prefix", "prefix_length", "prefix_source", "input_genomes", "total_bases", "retained_token_count", "retention_fraction", "feature_space_size", "observed_features", "matrix_rows", "matrix_columns", "matrix_nnz", "serialized_size_bytes"]:
            if column in group and group[column].notna().any():
                row[column] = group[column].dropna().iloc[0]
        timed_group = group[group["stage"].eq("overall")] if "stage" in group and group["stage"].eq("overall").any() else group
        durations = pd.to_numeric(timed_group.get("wall_time_seconds", pd.Series(dtype=float)), errors="coerce")
        rss = pd.to_numeric(group.get("peak_rss_bytes", pd.Series(dtype=float)), errors="coerce")
        row["runtime_seconds"] = durations.sum(min_count=1)
        row["peak_rss_bytes"] = rss.max()
        rows.append(row)
    return pd.DataFrame(rows)

resource_summary = summarize_resources(resources)
tradeoff = performance.merge(resource_summary, on=[c for c in ["condition_id", "model_name", "dataset", "phenotype", "representation", "total_k", "prefix", "prefix_length", "prefix_source"] if c in performance and c in resource_summary], how="left") if not performance.empty and not resource_summary.empty else pd.DataFrame()
if not tradeoff.empty and not paired_difference_summary.empty:
    tradeoff = tradeoff.merge(paired_difference_summary, on=["condition_id", "model_name", "dataset", "phenotype", "total_k", "prefix", "prefix_length"], how="left")
if not tradeoff.empty:
    full_resources = tradeoff[tradeoff["representation"].eq("full")][[c for c in ["model_name", "dataset", "phenotype", "total_k", "peak_rss_bytes", "runtime_seconds", "serialized_size_bytes"] if c in tradeoff]].drop_duplicates(["model_name", "dataset", "phenotype", "total_k"]).rename(columns={"peak_rss_bytes": "full_peak_rss_bytes", "runtime_seconds": "full_runtime_seconds", "serialized_size_bytes": "full_serialized_size_bytes"})
    tradeoff = tradeoff.merge(full_resources, on=["model_name", "dataset", "phenotype", "total_k"], how="left")
    for current, baseline, output in [("peak_rss_bytes", "full_peak_rss_bytes", "peak_rss_reduction_fraction"), ("runtime_seconds", "full_runtime_seconds", "runtime_reduction_fraction"), ("serialized_size_bytes", "full_serialized_size_bytes", "matrix_size_reduction_fraction")]:
        if current in tradeoff and baseline in tradeoff:
            tradeoff[output] = np.where(tradeoff["representation"].eq("prefix_filtered"), 1 - pd.to_numeric(tradeoff[current], errors="coerce") / pd.to_numeric(tradeoff[baseline], errors="coerce"), np.nan)
display(tradeoff.head(20))
if tradeoff.empty:
    print("Accuracy/resource trade-offs will appear once both metrics and resource tables are available.")

## 6. Gentamicin sensitivity/specificity and optional ResFinder overlay

Sensitivity and specificity are taken from the runner's explicit positive-class orientation. ResFinder is optional and is shown as an unpaired external reference unless its rows carry the same benchmark genome IDs.

In [ ]:
gentamicin = performance[performance["phenotype"].astype(str).str.lower().str.contains("gentamicin", na=False)].copy() if not performance.empty else pd.DataFrame()
if not gentamicin.empty and not paired_difference_summary.empty:
    gentamicin = gentamicin.merge(paired_difference_summary, on=["condition_id", "model_name", "dataset", "phenotype", "total_k", "prefix", "prefix_length"], how="left")
if not gentamicin.empty and not tradeoff.empty:
    gentamicin = gentamicin.merge(tradeoff[[c for c in ["condition_id", "model_name", "retention_fraction", "peak_rss_reduction_fraction", "runtime_reduction_fraction", "matrix_size_reduction_fraction"] if c in tradeoff]], on=["condition_id", "model_name"], how="left")
gentamicin_display = gentamicin[[c for c in ["condition_id", "model_name", "representation", "total_k", "prefix", "prefix_source", "balanced_accuracy_mean", "sensitivity_mean", "specificity_mean", "ci_lower", "ci_upper", "paired_difference_mean", "paired_ci_lower", "paired_ci_upper", "retention_fraction", "peak_rss_reduction_fraction", "runtime_reduction_fraction", "matrix_size_reduction_fraction"] if c in gentamicin]]
display(gentamicin_display)
if gentamicin.empty:
    print("Gentamicin sensitivity/specificity will appear after gentamicin metrics are saved.")

resfinder_summary = pd.DataFrame()
if prediction_metrics is not None and not resfinder.empty and {"genome_id", "true_label", "predicted_label"}.issubset(resfinder.columns):
    try:
        values = prediction_metrics(resfinder["true_label"], resfinder["predicted_label"], scores=resfinder["score"] if "score" in resfinder else None, positive_label=resfinder["positive_class"].dropna().iloc[0] if "positive_class" in resfinder and resfinder["positive_class"].notna().any() else None)
        resfinder_summary = pd.DataFrame([{"source": "ResFinder (external/unpaired unless IDs match)", **{k: v for k, v in values.items() if k in {"balanced_accuracy", "sensitivity", "specificity", "roc_auc", "n_samples"}}}])
    except Exception as error:
        print(f"ResFinder table found but could not be scored: {error}")
elif not resfinder.empty and prediction_metrics is None:
    print("ResFinder table found, but benchmark metrics dependencies are unavailable; skipping it safely.")
display(resfinder_summary)
if resfinder.empty:
    print("No ResFinder table found; its absence does not block the core benchmark.")

## 7. QC and supplement

The QC table checks condition/fold uniqueness, prediction coverage, resource units, and split leakage where cluster IDs are available. Confusion matrices and raw fold values are retained as exportable supplement tables.

In [ ]:
qc_rows = []
def qc(check: str, passed: bool, detail: str) -> None:
    qc_rows.append({"check": check, "passed": bool(passed), "detail": detail})

if metrics_ok:
    qc("unique model-condition-fold metrics", not metrics.duplicated(["model_name", "condition_id", "fold_id"]).any(), "No duplicate model/condition/fold score rows")
if predictions_ok:
    qc("unique model-condition-fold-genome predictions", not predictions.duplicated(["model_name", "condition_id", "fold_id", "genome_id"]).any(), "No duplicate model/condition out-of-fold predictions")
    qc("prediction labels present", predictions[["true_label", "predicted_label"]].notna().all().all(), "True and predicted labels are non-null")
if resources_ok:
    for column in ["wall_time_seconds", "cpu_time_seconds", "peak_rss_bytes", "serialized_size_bytes"]:
        if column in resources:
            values = pd.to_numeric(resources[column], errors="coerce")
            qc(f"non-negative {column}", bool((values.dropna() >= 0).all()), "Resource values use seconds or bytes as documented")
if not split_manifest.empty and {"genome_id", "fold_id", "cluster_id"}.issubset(split_manifest.columns):
    leakage = split_manifest.groupby("cluster_id")["fold_id"].nunique()
    qc("cluster does not cross folds", bool((leakage <= 1).all()), f"{int((leakage > 1).sum())} leaking clusters")

qc_table = pd.DataFrame(qc_rows)
fold_values = metrics.copy() if not metrics.empty else pd.DataFrame()
confusion_table = metrics[[c for c in ["condition_id", "model_name", "phenotype", "fold_id", "confusion_matrix", "classes"] if c in metrics]] if not metrics.empty else pd.DataFrame()
failed_conditions = failures.copy()
display(qc_table)
display(failed_conditions.head(20))
if qc_table.empty:
    print("QC will populate as saved outputs become available.")

## Figure definitions

The metadata below is exported with each figure. Full points are condition-level means across folds; filtered points/boxes use every seeded prefix condition. Error bars are 95% percentile intervals across folds. Box = Q1–Q3, center line = median, whiskers = most extreme values within 1.5×IQR, and individual points are overlaid. Chance lines are balanced-accuracy references of `1 / number_of_classes`.

In [ ]:
FIGURE_METADATA = {
    "Figure A": {"title": "Accuracy–resource trade-off", "chance_line": "balanced accuracy = 1 / n_classes", "points": "condition mean across folds; error bars are 95% fold percentile intervals", "boxes": None, "whiskers": None},
    "Figure B": {"title": "Sensitivity to prefix choice", "chance_line": "balanced accuracy = 1 / n_classes", "points": "one point per seeded prefix condition, mean across folds; historical prefix is a distinct point", "boxes": "Q1–Q3 of seeded prefix condition means", "whiskers": "most extreme observations within 1.5×IQR"},
    "Figure C": {"title": "Gentamicin full versus prefix-filtered", "chance_line": "balanced accuracy = 1 / n_classes", "points": "condition means across identical folds; error bars are seeded 95% fold intervals and paired full-minus-filtered intervals; external ResFinder is marked unpaired", "boxes": None, "whiskers": None},
    "Figure S1": {"title": "Prespecified-condition confusion matrices", "chance_line": "not applicable", "points": "each cell is a saved out-of-fold confusion-matrix count", "boxes": None, "whiskers": None},
    "Figure S2": {"title": "Fold-level paired performance differences", "chance_line": "zero means equal balanced accuracy", "points": "one full-minus-filtered difference per identical fold", "boxes": "Q1–Q3 across paired folds", "whiskers": "most extreme observations within 1.5×IQR"},
    "Figure S3": {"title": "Condition-level resource measurements", "chance_line": "not applicable", "points": "one condition summary for runtime, peak RSS, matrix size, or retained-token fraction", "boxes": None, "whiskers": None},
    "Figure S4": {"title": "Dataset class and organism composition", "chance_line": "not applicable", "points": "deduplicated genome counts; organism panel is shown when metadata is available", "boxes": None, "whiskers": None},
    "Figure S1": {"title": "Fold-level confusion matrices", "chance_line": None, "points": "one heatmap per saved condition/fold confusion matrix", "boxes": None, "whiskers": None},
}

def _caption(name: str) -> str:
    metadata = FIGURE_METADATA[name]
    return (f"{name}: {metadata['title']}. Chance reference: {metadata['chance_line']}. "
            f"Points: {metadata['points']}. "
            + (f"Boxes: {metadata['boxes']}; whiskers: {metadata['whiskers']}." if metadata["boxes"] else ""))

def _finish_figure(name: str, figure: Any) -> Any:
    figure.suptitle(_caption(name), fontsize=9, wrap=True)
    figure.tight_layout()
    return figure


## Figures A–C

These cells only plot the tidy summaries above. With no saved results they print a useful message and return without creating an empty publication figure.

In [ ]:
figures: dict[str, Any] = {}
if not PLOTTING_AVAILABLE:
    print("Figures skipped: install matplotlib and seaborn to render them.")
elif tradeoff.empty:
    print("Figure A skipped: no joined metrics/resource rows.")
else:
    phenotypes = [p for p in tradeoff["phenotype"].dropna().unique()]
    fig, axes = plt.subplots(max(1, len(phenotypes)), 1, figsize=(9, 4 * max(1, len(phenotypes))), squeeze=False)
    for axis, phenotype in zip(axes.flat, phenotypes):
        subset = tradeoff[tradeoff["phenotype"].eq(phenotype)].copy()
        subset["x_retention_fraction"] = pd.to_numeric(subset.get("retention_fraction"), errors="coerce")
        for representation, color in [("full", "tab:blue"), ("prefix_filtered", "tab:orange")]:
            points = subset[subset["representation"].eq(representation)].dropna(subset=["x_retention_fraction", "balanced_accuracy_mean"])
            if points.empty: continue
            axis.errorbar(points["x_retention_fraction"], points["balanced_accuracy_mean"], yerr=[points["balanced_accuracy_mean"] - points["ci_lower"], points["ci_upper"] - points["balanced_accuracy_mean"]], fmt="o", alpha=0.7, color=color, label=representation)
        chance = subset["chance_balanced_accuracy"].dropna().iloc[0] if subset["chance_balanced_accuracy"].notna().any() else 0.5
        axis.axhline(chance, color="0.45", linestyle="--", linewidth=1, label="chance")
        axis.set(title=str(phenotype), xlabel="Retained-token fraction", ylabel="Balanced accuracy")
        axis.legend()
    figures["Figure A"] = _finish_figure("Figure A", fig)
    display(fig)

In [ ]:
if not PLOTTING_AVAILABLE:
    print("Figure B skipped: install matplotlib and seaborn to render it.")
elif prefix_distribution.empty:
    print("Figure B skipped: no seeded prefix conditions.")
else:
    prefix_distribution = prefix_distribution.copy()
    prefix_distribution["facet"] = prefix_distribution["phenotype"].astype(str) + " | prefix length " + prefix_distribution["prefix_length"].astype(str)
    order = prefix_distribution["facet"].drop_duplicates().tolist()
    fig, axes = plt.subplots(max(1, len(order)), 1, figsize=(10, 3.5 * max(1, len(order))), squeeze=False)
    for axis, facet in zip(axes.flat, order):
        subset = prefix_distribution[prefix_distribution["facet"].eq(facet)]
        sns.boxplot(data=subset, x="total_k", y="balanced_accuracy_mean", ax=axis, color="#f4b183", whis=1.5, showfliers=False)
        sns.stripplot(data=subset, x="total_k", y="balanced_accuracy_mean", ax=axis, color="0.2", alpha=0.65, jitter=0.15)
        historical = prefix_summary[(prefix_summary["prefix_role"].eq("historical")) & (prefix_summary["phenotype"].astype(str) == str(subset["phenotype"].iloc[0])) & (prefix_summary["prefix_length"] == subset["prefix_length"].iloc[0])]
        if not historical.empty:
            sns.scatterplot(data=historical, x="total_k", y="balanced_accuracy_mean", ax=axis, color="crimson", marker="D", s=55, label="historical prefix")
        chance = subset["chance_balanced_accuracy"].dropna().iloc[0] if "chance_balanced_accuracy" in subset and subset["chance_balanced_accuracy"].notna().any() else 0.5
        axis.axhline(chance, color="0.45", linestyle="--", linewidth=1, label="chance")
        axis.set(title=facet, xlabel="Total k-mer length", ylabel="Balanced accuracy")
    figures["Figure B"] = _finish_figure("Figure B", fig)
    display(fig)

In [ ]:
if not PLOTTING_AVAILABLE:
    print("Figure C skipped: install matplotlib and seaborn to render it.")
elif gentamicin.empty:
    print("Figure C skipped: no gentamicin metrics.")
else:
    fig, axes = plt.subplots(1, 4, figsize=(20, 4))
    plot_data = gentamicin.copy()
    plot_data["condition_label"] = plot_data["representation"].astype(str) + " k=" + plot_data["total_k"].astype(str)
    for axis, metric, title, low, high in [(axes[0], "balanced_accuracy_mean", "Balanced accuracy", "ci_lower", "ci_upper"), (axes[1], "sensitivity_mean", "Sensitivity", "sensitivity_ci_lower", "sensitivity_ci_upper")]:
        sns.stripplot(data=plot_data, x="total_k", y=metric, hue="representation", dodge=True, ax=axis, alpha=0.75)
        for _, point in plot_data.dropna(subset=[metric, low, high]).iterrows():
            axis.errorbar(point["total_k"], point[metric], yerr=[[point[metric] - point[low]], [point[high] - point[metric]]], fmt="none", color="0.25", capsize=3, alpha=0.7)
        axis.set(title=title, xlabel="Total k-mer length", ylabel=title)
    axis = axes[2]
    sns.stripplot(data=plot_data, x="total_k", y="paired_difference_mean", hue="representation", dodge=True, ax=axis, alpha=0.75)
    for _, point in plot_data.dropna(subset=["paired_difference_mean", "paired_ci_lower", "paired_ci_upper"]).iterrows():
        axis.errorbar(point["total_k"], point["paired_difference_mean"], yerr=[[point["paired_difference_mean"] - point["paired_ci_lower"]], [point["paired_ci_upper"] - point["paired_difference_mean"]]], fmt="none", color="0.25", capsize=3, alpha=0.7)
    axis.axhline(0, color="0.45", linestyle="--", linewidth=1)
    axis.set(title="Full − filtered balanced accuracy", xlabel="Total k-mer length", ylabel="Difference")
    reductions = plot_data[plot_data["representation"].eq("prefix_filtered")].melt(id_vars=["total_k"], value_vars=[c for c in ["peak_rss_reduction_fraction", "runtime_reduction_fraction", "matrix_size_reduction_fraction"] if c in plot_data], var_name="resource", value_name="reduction").dropna()
    if reductions.empty:
        axes[3].text(0.5, 0.5, "No resource reductions saved", ha="center", va="center")
    else:
        sns.stripplot(data=reductions, x="total_k", y="reduction", hue="resource", dodge=True, ax=axes[3], alpha=0.75)
        axes[3].axhline(0, color="0.45", linestyle="--", linewidth=1)
    axes[3].set(title="Resource reduction", xlabel="Total k-mer length", ylabel="Fraction reduced")
    figures["Figure C"] = _finish_figure("Figure C", fig)
    display(fig)

In [ ]:
# Essential QC supplement: saved confusion matrices, one heatmap per condition/fold.
if not PLOTTING_AVAILABLE:
    print("Figure S1 skipped: install matplotlib and seaborn to render it.")
elif confusion_table.empty:
    print("Figure S1 skipped: no saved confusion matrices.")
else:
    matrices = confusion_table.head(12)
    fig, axes = plt.subplots(1, len(matrices), figsize=(3.2 * len(matrices), 3), squeeze=False)
    for axis, (_, row) in zip(axes.flat, matrices.iterrows()):
        values = np.asarray(json.loads(row["confusion_matrix"]))
        sns.heatmap(values, annot=True, fmt="g", cbar=False, ax=axis, cmap="Blues")
        axis.set(title=f"{row.get('condition_id', '')} fold {row.get('fold_id', '')}", xlabel="Predicted", ylabel="True")
    figures["Figure S1"] = _finish_figure("Figure S1", fig)
    display(fig)

In [ ]:
# Supplementary paired-fold, resource, and dataset-composition figures.
paired_fold_rows = []
if not metrics.empty and {"condition_id", "fold_id", "balanced_accuracy"}.issubset(metrics.columns):
    for condition_id, filtered in metrics[metrics["representation"].eq("prefix_filtered")].groupby("condition_id"):
        first = filtered.iloc[0]
        full = metrics[(metrics["representation"].eq("full")) & (metrics["dataset"].eq(first["dataset"])) & (metrics["phenotype"].eq(first["phenotype"])) & (metrics["total_k"].eq(first["total_k"]))]
        aligned = full[["fold_id", "balanced_accuracy"]].merge(filtered[["fold_id", "balanced_accuracy"]], on="fold_id", suffixes=("_full", "_filtered"))
        for _, row in aligned.iterrows():
            paired_fold_rows.append({"condition_id": condition_id, "phenotype": first["phenotype"], "prefix_length": first["prefix_length"], "fold_id": row["fold_id"], "difference": row["balanced_accuracy_full"] - row["balanced_accuracy_filtered"]})
paired_fold_values = pd.DataFrame(paired_fold_rows)
if not PLOTTING_AVAILABLE or paired_fold_values.empty:
    print("Figure S2 skipped: no paired fold values.")
else:
    fig, axis = plt.subplots(figsize=(10, max(4, 0.7 * paired_fold_values["phenotype"].nunique())))
    sns.boxplot(data=paired_fold_values, x="difference", y="phenotype", whis=1.5, showfliers=False, color="#d9eaf7", ax=axis)
    sns.stripplot(data=paired_fold_values, x="difference", y="phenotype", hue="prefix_length", alpha=0.45, ax=axis)
    axis.axvline(0, color="0.35", linestyle="--", linewidth=1)
    axis.set(xlabel="Full − filtered balanced accuracy", ylabel="Phenotype")
    figures["Figure S2"] = _finish_figure("Figure S2", fig)
    display(fig)

resource_columns = [c for c in ["runtime_seconds", "peak_rss_bytes", "serialized_size_bytes", "retention_fraction"] if c in resource_summary]
if not PLOTTING_AVAILABLE or resource_summary.empty or not resource_columns:
    print("Figure S3 skipped: no condition resource summaries.")
else:
    resource_plot = resource_summary.melt(id_vars=[c for c in ["condition_id", "phenotype", "representation"] if c in resource_summary], value_vars=resource_columns, var_name="resource", value_name="value").dropna()
    fig, axes = plt.subplots(2, 2, figsize=(14, 9), squeeze=False)
    for axis, resource in zip(axes.flat, resource_columns):
        subset = resource_plot[resource_plot["resource"].eq(resource)]
        sns.stripplot(data=subset, x="representation", y="value", hue="phenotype", alpha=0.6, ax=axis)
        axis.set(title=resource.replace("_", " "), xlabel="Representation", ylabel=resource)
    for axis in axes.flat[len(resource_columns):]: axis.set_visible(False)
    figures["Figure S3"] = _finish_figure("Figure S3", fig)
    display(fig)

composition = predictions[[c for c in ["dataset", "phenotype", "genome_id", "true_label"] if c in predictions]].drop_duplicates() if not predictions.empty else pd.DataFrame()
organism_columns = [c for c in ["species", "organism", "genus", "organism_name"] if c in split_manifest]
if not PLOTTING_AVAILABLE or composition.empty or "true_label" not in composition:
    print("Figure S4 skipped: no prediction-level class composition.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    class_counts = composition.groupby(["phenotype", "true_label"], dropna=False).size().rename("genomes").reset_index()
    sns.barplot(data=class_counts, x="phenotype", y="genomes", hue="true_label", ax=axes[0])
    axes[0].tick_params(axis="x", rotation=45)
    axes[0].set(title="Class composition", xlabel="Phenotype", ylabel="Genomes")
    if organism_columns:
        organism = organism_columns[0]
        organism_counts = split_manifest[organism].dropna().astype(str).value_counts().head(15).rename_axis(organism).rename("genomes").reset_index()
        sns.barplot(data=organism_counts, x="genomes", y=organism, color="#80b1d3", ax=axes[1])
        axes[1].set(title=f"Top {organism} groups", xlabel="Genomes", ylabel=organism)
    else:
        axes[1].text(0.5, 0.5, "Organism metadata unavailable", ha="center", va="center")
        axes[1].set_axis_off()
    figures["Figure S4"] = _finish_figure("Figure S4", fig)
    display(fig)

## 8. Export publication-ready figures and summary tables

Exports are opt-in via `REVIEWER_BENCHMARK_EXPORT=1`; this keeps the notebook read-only by default. The exported tables are the exact tidy inputs used for figures and QC, together with figure-caption metadata.

In [ ]:
export_tables = {
    "dataset_summary.csv": dataset_table,
    "performance_summary.csv": performance,
    "prefix_summary.csv": prefix_summary,
    "resource_summary.csv": resource_summary,
    "tradeoff_summary.csv": tradeoff,
    "gentamicin_summary.csv": gentamicin_display,
    "fold_metrics.csv": fold_values,
    "confusion_matrices.csv": confusion_table,
    "qc.csv": qc_table,
    "failed_conditions.csv": failed_conditions,
}

if EXPORT_OUTPUTS:
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    for filename, table in export_tables.items():
        table.to_csv(EXPORT_DIR / filename, index=False)
    (EXPORT_DIR / "figure_metadata.json").write_text(json.dumps({name: {**metadata, "caption": _caption(name)} for name, metadata in FIGURE_METADATA.items()}, indent=2) + "\n")
    for name, figure in figures.items():
        stem = name.lower().replace(" ", "_")
        figure.savefig(EXPORT_DIR / f"{stem}.png", dpi=300, bbox_inches="tight")
        figure.savefig(EXPORT_DIR / f"{stem}.svg", bbox_inches="tight")
    print(f"Exported {len(export_tables)} tables and {len(figures)} figures to {EXPORT_DIR.resolve()}")
else:
    print("Exports disabled. Set REVIEWER_BENCHMARK_EXPORT=1 to write CSV, PNG, SVG, and caption metadata files.")